# Golden Hour · GUARDIAN v2 — ALL-CURRENCY, LOW-DISRUPTION FREEZE OPTIMISER

**Use this as a separate, review-only experiment.** Do not edit the team's earlier notebook or overwrite `freeze_plan.csv`.

What this version fixes:
- Every account freeze is evaluated **across all currencies**, not just US Dollar transactions.
- Currency conversions are explicit, auditable and never silently treated as USD.
- Suspicious-benefit outliers may be capped **for selection only**; collateral and gross-exposure constraints use **full uncapped converted value**.
- An exact binary OR formulation accounts for overlapping frozen endpoints and self-transfers once.
- Optional gross, proxy-collateral and per-account exposure safeguards are enforced on the **same full dataset**.
- Reports include comparable historical baselines, transaction counts, label-only retrospective evaluation, and currency-by-currency audit.
- A separate output directory is used. No freeze is executed. Decisions require authorised human review.

**Key limitations:** This is retrospective analysis of a *synthetic AML dataset*. `account_risk.csv` may contain full-history information; the optimiser is **not a validated online model**. Risk-score products are **uncalibrated suspicion proxies**, not transaction probabilities. Currency rates are approximations, and a payment being touched is not evidence of permanently lost money.

Run the notebook **Kernel → Restart Kernel and Run All Cells** from `/home/ibm-user/shared`, after confirming Step 1 and Step 2 input files exist. It is designed for the shared LinuxONE VM. Initial runtime may be tens of seconds due to bounded MILP solves.

## 1. Configuration — confirm these assumptions before using results

Budgets are exploratory **risk-policy assumptions**, not validated banking thresholds. Change them based on a specified policy **before** examining labelled performance. The default selected plan minimises collateral proxy while maintaining at least **50% of feasible maximum suspicious-benefit proxy** under these budgets; it does not assert 50% actual laundering capture. Very restrictive budgets can still yield zero recommendations.

**Check with Abdurub:** `amount_paid` must be paired with its **Payment Currency**, not Receiving Currency. Exchange rates here are deliberately frozen approximations for a prototype. For audited financial amounts, use date-appropriate verified FX conversion.

In [ ]:
from __future__ import annotations
import os
for _var in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, '1')
from dataclasses import dataclass, replace, asdict
from pathlib import Path
import hashlib, json, time
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix, vstack
try:
    from scipy.optimize import milp, Bounds, LinearConstraint
    HAVE_MILP = True
except ImportError:
    HAVE_MILP = False

# Approximate USD for one original-currency unit, a fixed prototype reference.
# In particular, this Bitcoin conversion is not a live or time-specific rate.
FX_TO_USD = {
    'US Dollar': 1.0, 'Euro': 1.08, 'UK Pound': 1.27, 'Swiss Franc': 1.12,
    'Yen': 0.0067, 'Yuan': 0.14, 'Rupee': 0.012, 'Ruble': 0.011,
    'Canadian Dollar': 0.73, 'Australian Dollar': 0.66, 'Mexican Peso': 0.055,
    'Brazil Real': 0.18, 'Saudi Riyal': 0.27, 'Shekel': 0.27,
    'Bitcoin': 60000.0,
}

@dataclass(frozen=True)
class Config:
    k: int = 10
    min_account_score: float = 0.10
    max_candidates: int = 65
    shortlist_each: int = 14
    candidate_lambdas: tuple = (0., 1., 3., 5., 10., 20., 50.)
    frontier_lambdas: tuple = (0., 2., 5., 10., 20., 50.)
    chosen_lambda: float = 20.0  # tested as a sensitivity setting, not default selection
    chosen_policy: str = 'min_collateral_with_min_benefit'
    chosen_target_fraction: float = 0.50  # of max feasible benefit PROXY; predefined, not chosen from labels
    coverage_targets: tuple = (0.50, 0.75)  # of max *predicted proxy* coverage, not labels
    benefit_cap_quantile: float = 0.999
    min_proxy_legitimate_fraction: float = 0.01
    # Conservative retrospective exposure controls: both cover ALL currencies.
    max_account_gross_usd: float | None = 100_000_000.0
    max_account_single_transfer_usd: float | None = 75_000_000.0
    max_portfolio_gross_usd: float | None = 250_000_000.0
    max_portfolio_proxy_collateral_usd: float | None = 15_000_000.0
    max_portfolio_touched_tx: int | None = 2500
    hub_outgoing_review: int = 10000
    exclude_extreme_hubs: bool = True
    mip_seconds: float = 5.0  # max per model; multiple models are tested
    mip_rel_gap: float = 0.015

SERVER_SHARED = Path('/home/ibm-user/shared')
SHARED = SERVER_SHARED if SERVER_SHARED.is_dir() else Path.cwd()
INPUT_PATH = SHARED / 'clean_transactions.csv'
RISK_PATH = SHARED / 'account_risk.csv'
PRIOR_PLAN_PATH = SHARED / 'freeze_plan.csv'
PRIOR_GUARDIAN_PATH = SHARED / 'guardian_candidate_outputs' / 'freeze_plan_candidate.csv'
OUTPUT_DIR = SHARED / 'guardian_v2_outputs'  # Never overwrites existing team's plan
CFG = Config()

print('Input transactions:', INPUT_PATH)
print('Input risk scores:', RISK_PATH)
print('New output directory:', OUTPUT_DIR)
print('SciPy mixed-integer solver:', HAVE_MILP)

## 2. Load ALL transactions once, map accounts to integers, validate every currency

**Labels are loaded only for an independent retrospective audit**; they must not enter candidate construction, constraints, optimisation or explanations. Unknown/missing currency, invalid amounts, unknown account risk and duplicate transaction IDs cause an error instead of silently mispricing transfers.

In [ ]:
@dataclass
class Data:
    accounts: np.ndarray
    risks: np.ndarray
    sender: np.ndarray
    receiver: np.ndarray
    dollars: np.ndarray
    original_amount: np.ndarray
    currency: np.ndarray
    labelled: np.ndarray
    timestamp: np.ndarray
    from_bank: np.ndarray
    to_bank: np.ndarray
    tx_id: np.ndarray
    n_out: np.ndarray
    n_in: np.ndarray
    incident_gross: np.ndarray
    max_incident: np.ndarray
    tx_file_rows: int
    risk_file_rows: int
    fx_rates: dict


def incident_sum(f, t, w, n):
    '''Exact sum of weights touching each account; self-payments counted only once.'''
    ans = np.bincount(f, weights=w, minlength=n)
    other = f != t
    ans += np.bincount(t[other], weights=w[other], minlength=n)
    return ans


def load_data(transactions_path, risk_path, fx_rates=FX_TO_USD):
    transactions_path, risk_path = Path(transactions_path), Path(risk_path)
    if not transactions_path.is_file() or not risk_path.is_file():
        raise FileNotFoundError('Upload both clean_transactions.csv and account_risk.csv into /shared first.')
    required = {'tx_id', 'timestamp', 'from_acct', 'to_acct', 'amount_paid', 'currency', 'is_laundering'}
    optional = {'from_bank', 'to_bank', 'payment_format'}
    tx = pd.read_csv(transactions_path, usecols=lambda c: c in required | optional,
                     dtype={'from_acct': 'string', 'to_acct': 'string', 'currency': 'string',
                            'from_bank': 'string', 'to_bank': 'string'})
    if missing := sorted(required - set(tx.columns)):
        raise ValueError(f'Missing transaction columns: {missing}')
    for name in ('from_acct', 'to_acct', 'currency'):
        if tx[name].isna().any() or tx[name].astype('string').str.strip().eq('').any():
            raise ValueError(f'Blank/missing {name} values; do not auto-repair IDs or currency.')
    if tx['tx_id'].isna().any() or tx['tx_id'].duplicated().any():
        raise ValueError('Missing or duplicate tx_id: cannot safely evaluate distinct transfers.')
    raw_amt = pd.to_numeric(tx.amount_paid, errors='coerce').to_numpy(dtype=float)
    if not (np.isfinite(raw_amt).all() and (raw_amt >= 0).all()):
        raise ValueError('amount_paid has negative, missing or infinite values.')
    labels = pd.to_numeric(tx.is_laundering, errors='coerce').to_numpy(dtype=float)
    if not (np.isfinite(labels).all() and np.isin(labels, [0.,1.]).all()):
        raise ValueError('is_laundering must be 0 or 1 throughout.')
    timestamps = pd.to_datetime(tx.timestamp, errors='coerce', utc=True)
    if timestamps.isna().any():
        raise ValueError('Invalid timestamps. This notebook will not invent or forward-fill them.')
    found_currency = sorted(tx.currency.unique().tolist())
    missing_currency = sorted(set(found_currency) - set(fx_rates))
    if missing_currency:
        raise ValueError(f'No explicit FX assumption for: {missing_currency}; update FX_TO_USD or stop.')
    rates = tx.currency.map(fx_rates).to_numpy(dtype=float)
    if not (np.isfinite(rates).all() and (rates > 0).all()):
        raise ValueError('Invalid FX assumptions.')
    dollars = raw_amt * rates
    if not np.isfinite(dollars).all():
        raise ValueError('Converted values overflow; inspect large input amounts.')
    risk = pd.read_csv(risk_path, usecols=['acct', 'risk_score'], dtype={'acct': 'string'})
    if risk.acct.isna().any() or risk.acct.duplicated().any():
        raise ValueError('Duplicate or null IDs in account_risk.csv')
    risk['risk_score'] = pd.to_numeric(risk.risk_score, errors='coerce')
    if risk.risk_score.isna().any() or not risk.risk_score.between(0,1).all():
        raise ValueError('Risk scores are missing or outside the interval [0, 1].')
    # Dense codes; only pseudonymous identifiers are used.
    from_account = tx.from_acct.to_numpy(dtype=str)
    to_account = tx.to_acct.to_numpy(dtype=str)
    accounts = np.unique(np.concatenate([from_account, to_account]))
    f = np.searchsorted(accounts, from_account).astype(np.int32)
    t = np.searchsorted(accounts, to_account).astype(np.int32)
    r = risk.set_index('acct').risk_score.reindex(accounts)
    coverage = r.notna().mean()
    if not r.notna().all():
        raise ValueError(f'Only {coverage:.3%} risk-score coverage; missing scores must not be zero-filled.')
    n = len(accounts)
    nout = np.bincount(f, minlength=n)
    nin = np.bincount(t, minlength=n)
    gross = incident_sum(f, t, dollars, n)
    max_incident = np.zeros(n, dtype=np.float64)
    np.maximum.at(max_incident, f, dollars)
    np.maximum.at(max_incident, t, dollars)
    send_bank = tx.from_bank.fillna('unknown').to_numpy(str) if 'from_bank' in tx else np.repeat('unknown', len(tx))
    recv_bank = tx.to_bank.fillna('unknown').to_numpy(str) if 'to_bank' in tx else np.repeat('unknown', len(tx))
    data = Data(accounts, r.to_numpy(float), f, t, dollars, raw_amt,
                tx.currency.to_numpy(str), labels.astype(np.int8), timestamps.to_numpy(),
                send_bank, recv_bank, tx.tx_id.to_numpy(), nout, nin, gross,
                max_incident, len(tx), len(risk), dict(fx_rates))
    del tx, risk
    print(f'ALL-CURRENCY data: {data.tx_file_rows:,} transfers; {n:,} accounts; risk coverage {coverage:.2%}; {len(found_currency)} currencies')
    print('Currencies:', ', '.join(found_currency))
    print('WARNING: Approximate FX assumptions and full-history risk scores; retrospective only.')
    return data

## 3. Candidate selection and a correct full-exposure portfolio model

For each transfer, `q = sender_risk × receiver_risk` is an **uncalibrated ranking heuristic**. The *benefit proxy* may use capped USD values so a whale transfer does not dominate the objective. **Every cost constraint uses full USD value** (plus a conservative minimum proxy legitimate share), even if the benefit is clipped.

A whole-account freeze touches every incident transaction. The binary coverage variable for each compressed pair of candidate endpoints equals their logical OR, including when the transfer also involves an unselected account. Both endpoints selected still count only once.

In [ ]:
@dataclass
class Problem:
    data: Data
    cfg: Config
    candidates: np.ndarray
    group_nodes: list
    group_benefit: np.ndarray
    group_cost: np.ndarray
    group_gross: np.ndarray
    group_count: np.ndarray
    risk_proxy: np.ndarray
    benefit_cap: float
    base_A: object
    base_lower: np.ndarray
    base_upper: np.ndarray
    eligible_count: int


def build_problem(d: Data, cfg: Config):
    if cfg.k < 1 or cfg.max_candidates < 1 or cfg.shortlist_each < 1:
        raise ValueError('Invalid action or shortlist budget.')
    if not (0 < cfg.benefit_cap_quantile <= 1 and 0 <= cfg.min_proxy_legitimate_fraction <= 1):
        raise ValueError('Invalid amount cap/floor assumptions.')
    f, t, n = d.sender, d.receiver, len(d.accounts)
    q = np.clip(d.risks[f] * d.risks[t], 0., 1.)
    cap = float(np.quantile(d.dollars, cfg.benefit_cap_quantile))
    b = np.minimum(d.dollars, cap) * q
    c = d.dollars * np.maximum(1. - q, cfg.min_proxy_legitimate_fraction)
    gross = d.dollars
    allowed = (d.risks >= cfg.min_account_score) & ((d.n_in + d.n_out) > 0)
    if cfg.max_account_gross_usd is not None:
        allowed &= d.incident_gross <= cfg.max_account_gross_usd
    if cfg.max_account_single_transfer_usd is not None:
        allowed &= d.max_incident <= cfg.max_account_single_transfer_usd
    if cfg.exclude_extreme_hubs:
        allowed &= d.n_out < cfg.hub_outgoing_review
    eligible = np.flatnonzero(allowed)
    if not len(eligible):
        print('ABSTAIN: no accounts pass the configured all-currency exposure safeguards.')
    # Distinct shortlist criteria; never consult true labels.
    ranks = {}
    base_b = incident_sum(f, t, b, n)
    base_c = incident_sum(f, t, c, n)
    def contribute(values, limit, weight=1.0):
        scores = values.copy()
        scores[~allowed] = -np.inf
        take = min(limit, len(eligible))
        if take <= 0: return
        # Stable sort by score (ties by account code), no random sampling.
        ii = np.argsort(-scores, kind='stable')[:take]
        for rank, x in enumerate(ii):
            if np.isfinite(scores[x]):
                ranks[int(x)] = ranks.get(int(x), 0.0) + weight / (rank + 1)
    for lam in cfg.candidate_lambdas:
        contribute(base_b - lam*base_c, cfg.shortlist_each)
    contribute(base_b, cfg.shortlist_each, 1.4)
    contribute(base_b / np.maximum(base_c, 1.), cfg.shortlist_each, 1.0)
    risk_order = np.full(n, -np.inf)
    risk_order[allowed] = d.risks[allowed]
    contribute(risk_order, cfg.shortlist_each, .5)
    candidates = np.asarray(sorted(ranks, key=lambda x: (-ranks[x], x))[:cfg.max_candidates], dtype=np.int32)
    m = len(candidates)
    # Group transfers exactly by which shortlist endpoints they touch.
    mapping = np.full(n, m, dtype=np.int32)  # m is sentinel for not shortlisted
    mapping[candidates] = np.arange(m, dtype=np.int32)
    first, second = mapping[f], mapping[t]
    low, high = np.minimum(first, second), np.maximum(first, second)
    valid = low < m
    encoded = low[valid].astype(np.int64) * (m + 1) + high[valid]
    codes, inv = np.unique(encoded, return_inverse=True)
    group_nodes = []
    for code in codes:
        x, y = int(code // (m+1)), int(code % (m+1))
        group_nodes.append((x,) if y == m or x == y else (x, y))
    G = len(codes)
    gb = np.bincount(inv, weights=b[valid], minlength=G)
    gc = np.bincount(inv, weights=c[valid], minlength=G)
    gg = np.bincount(inv, weights=gross[valid], minlength=G)
    gn = np.bincount(inv, minlength=G).astype(float)
    # A compact sparse MILP constraint matrix. Each group y is exactly OR(x endpoints).
    rows, cols, vals, lo, hi = [], [], [], [], []
    def add(items, lower=-np.inf, upper=np.inf):
        idx = len(lo)
        for cidx, coeff in items:
            rows.append(idx); cols.append(cidx); vals.append(coeff)
        lo.append(lower); hi.append(upper)
    add([(i,1.) for i in range(m)], upper=min(cfg.k, m))
    for g, nodes in enumerate(group_nodes):
        y = m + g
        for x in nodes:
            add([(y,1.),(x,-1.)], lower=0.)
        add([(y,1.)] + [(x,-1.) for x in nodes], upper=0.)
    A = coo_matrix((np.asarray(vals,float),(rows,cols)), shape=(len(lo),m+G)).tocsr()
    print(f'Eligibility: {len(eligible):,} / {n:,} accounts; shortlist {m}; coverage groups {G:,}')
    print(f'Benefit clipped above ${cap:,.0f} USD per transfer; collateral/gross NEVER clipped.')
    return Problem(d, cfg, candidates, group_nodes, gb, gc, gg, gn, q, cap, A,
                   np.asarray(lo,float), np.asarray(hi,float), len(eligible))

## 4. Constrained MILP and safe greedy fallback

The solver maximises benefit proxy (or benefit minus a configurable penalty for collateral proxy), **subject to** max 10 freezes, the full-value gross-exposure cap, the full-value proxy-collateral cap, and the transfer-count cap. It may choose fewer or zero accounts.

If SciPy MILP is absent or fails to return a feasible incumbent, a clearly labelled greedy approximation is used; **no approximation guarantee** is claimed for the signed, constrained problem. MILP optimality claims are restricted to its candidate shortlist and explicit proxy model.

In [ ]:
def coverage_by_group(p, selected):
    chosen = set(int(i) for i in selected)
    selected_local = {j for j, i in enumerate(p.candidates) if int(i) in chosen}
    return np.fromiter((any(j in selected_local for j in nodes) for nodes in p.group_nodes),
                       count=len(p.group_nodes), dtype=bool)


def plan_numbers(p, selected):
    cover = coverage_by_group(p, selected)
    return (float(p.group_benefit[cover].sum()), float(p.group_cost[cover].sum()),
            float(p.group_gross[cover].sum()), int(p.group_count[cover].sum()))


def meets_limits(p, numbers):
    _, c, g, tcount = numbers
    cfg = p.cfg
    tol = 1e-7
    return ((cfg.max_portfolio_proxy_collateral_usd is None or c <= cfg.max_portfolio_proxy_collateral_usd*(1+tol))
        and (cfg.max_portfolio_gross_usd is None or g <= cfg.max_portfolio_gross_usd*(1+tol))
        and (cfg.max_portfolio_touched_tx is None or tcount <= cfg.max_portfolio_touched_tx))


def greedy_fallback(p, mode, lam=0., target=0., message='heuristic approximation'):
    m = len(p.candidates)
    rows = [[] for _ in range(m)]
    for g,nodes in enumerate(p.group_nodes):
        for i in nodes: rows[i].append(g)
    rows = [np.asarray(z, dtype=np.int32) for z in rows]
    covered = np.zeros(len(p.group_nodes), bool)
    chosen_local = []
    current_b = current_c = current_g = 0.
    current_n = 0
    for _ in range(min(p.cfg.k,m)):
        options = []
        for i, idx in enumerate(rows):
            if i in chosen_local: continue
            new_idx = idx[~covered[idx]]
            db = float(p.group_benefit[new_idx].sum())
            dc = float(p.group_cost[new_idx].sum())
            dg = float(p.group_gross[new_idx].sum())
            dn = int(p.group_count[new_idx].sum())
            if not meets_limits(p, (current_b+db,current_c+dc,current_g+dg,current_n+dn)):
                continue
            if mode == 'max_benefit': score = db
            elif mode == 'penalised': score = db - lam * dc
            else: score = db / max(dc,1.)
            options.append((score, -i, i, db, dc, dg, dn))
        if not options: break
        score, _, i, db, dc, dg, dn = max(options)
        if score <= 0: break
        chosen_local.append(i)
        covered[rows[i]] = True
        current_b += db; current_c += dc; current_g += dg; current_n += dn
        if mode == 'min_cost_target' and current_b + 1e-5 >= target: break
    chosen = p.candidates[np.asarray(chosen_local,dtype=np.int32)]
    nums = plan_numbers(p, chosen)
    if mode == 'min_cost_target' and nums[0] < target - max(1.,target*1e-5):
        return None
    assert meets_limits(p, nums)
    return dict(mode=mode, method='', selected=chosen, n_selected=len(chosen),
                proxy_benefit=nums[0], proxy_collateral=nums[1], gross_touched=nums[2],
                tx_touched=nums[3], status='HEURISTIC / NO OPTIMALITY CLAIM: '+message,
                status_code=-1, mip_gap=None, runtime_seconds=0.,
                penalty_weight=lam, target_proxy=target)


def solve(p, mode, lam=0., target=0.):
    m, G = len(p.candidates), len(p.group_nodes)
    if m == 0:
        if mode == 'min_cost_target' and target > 0: return None
        return dict(mode=mode, method='', selected=np.array([],dtype=np.int32), n_selected=0,
                    proxy_benefit=0., proxy_collateral=0., gross_touched=0., tx_touched=0,
                    status='ABSTAIN: no candidate meets the configured safeguards',
                    status_code=-2, mip_gap=None, runtime_seconds=0.,
                    penalty_weight=lam, target_proxy=target)
    if not HAVE_MILP:
        return greedy_fallback(p, mode, lam, target, 'SciPy MILP unavailable')
    # Coefficients in millions of USD to reduce numerical range.
    scale = 1_000_000.
    cvec = np.zeros(m+G, dtype=float)
    if mode == 'max_benefit':
        cvec[m:] = -p.group_benefit/scale
    elif mode == 'penalised':
        cvec[m:] = (-p.group_benefit+lam*p.group_cost)/scale
    elif mode == 'min_cost_target':
        cvec[m:] = (p.group_cost + 0.00001*p.group_gross)/scale
        cvec[:m] = 1e-4  # stable preference for fewer actions in ties
    else:
        raise ValueError('Unknown solve mode: '+mode)
    rows = [p.base_A]
    lows = [p.base_lower]
    highs = [p.base_upper]
    def add_limit(weight, value, relation, row_scale=scale):
        row = csr_matrix((weight/row_scale, (np.zeros(G,dtype=int),np.arange(m,m+G))),shape=(1,m+G))
        rows.append(row)
        if relation == 'max': lows.append(np.array([-np.inf])); highs.append(np.array([value/row_scale]))
        else: lows.append(np.array([value/row_scale])); highs.append(np.array([np.inf]))
    if p.cfg.max_portfolio_proxy_collateral_usd is not None:
        add_limit(p.group_cost, p.cfg.max_portfolio_proxy_collateral_usd, 'max')
    if p.cfg.max_portfolio_gross_usd is not None:
        add_limit(p.group_gross, p.cfg.max_portfolio_gross_usd, 'max')
    if p.cfg.max_portfolio_touched_tx is not None:
        # Count constraints stay in transaction units, not million-dollar units.
        add_limit(p.group_count, p.cfg.max_portfolio_touched_tx, 'max', row_scale=1.)
    if mode == 'min_cost_target':
        add_limit(p.group_benefit, target, 'min')
    A = vstack(rows, format='csr')
    low, high = np.concatenate(lows), np.concatenate(highs)
    start = time.perf_counter()
    try:
        result = milp(cvec, integrality=np.ones(m+G,dtype=np.int8),
                      bounds=Bounds(np.zeros(m+G),np.ones(m+G)),
                      constraints=LinearConstraint(A,low,high),
                      options={'presolve':True,'time_limit':p.cfg.mip_seconds,
                               'mip_rel_gap':p.cfg.mip_rel_gap})
    except (ValueError, RuntimeError) as exc:
        return greedy_fallback(p,mode,lam,target,'MILP error: '+str(exc)[:140])
    duration = time.perf_counter() - start
    if result.x is None:
        return greedy_fallback(p,mode,lam,target,'MILP no incumbent: '+str(result.message)[:100])
    local = np.flatnonzero(result.x[:m] > .5)
    selected = p.candidates[local]
    measured = plan_numbers(p, selected)
    if len(selected)>p.cfg.k or not meets_limits(p,measured):
        raise AssertionError('The solver returned a policy-violating plan; stop and inspect.')
    if mode == 'min_cost_target' and measured[0] < target - max(1.,target*1e-5):
        raise AssertionError('Solver returned a solution below the requested proxy target.')
    # Verify solver y variables independently from selected endpoint coverage.
    y = coverage_by_group(p,selected)
    if not np.allclose(result.x[m:], y.astype(float), atol=1e-5):
        raise AssertionError('Coverage variables disagree with endpoint union.')
    return dict(mode=mode,method='',selected=selected,n_selected=len(selected),
                proxy_benefit=measured[0],proxy_collateral=measured[1],
                gross_touched=measured[2],tx_touched=measured[3],
                status=str(result.message),status_code=int(result.status),
                mip_gap=(float(result.mip_gap) if getattr(result,'mip_gap',None) is not None else None),
                runtime_seconds=duration,penalty_weight=lam,target_proxy=target)


def run_frontier(p):
    start = time.perf_counter()
    points = []
    best = solve(p, 'max_benefit')
    if best is None: raise ValueError('No feasible plan, including the empty plan: check constraints.')
    best['method'] = 'Maximum proxy benefit within safety budgets'
    points.append(best)
    for lam in p.cfg.frontier_lambdas:
        sol = solve(p, 'penalised', lam=float(lam))
        if sol is None: continue
        sol['method'] = f'Proxy penalty {lam:g} (constrained)'
        points.append(sol)
    if p.cfg.chosen_lambda not in p.cfg.frontier_lambdas:
        sol = solve(p, 'penalised', lam=p.cfg.chosen_lambda)
        if sol is not None:
            sol['method'] = f'Proxy penalty {p.cfg.chosen_lambda:g} (constrained)'
            points.append(sol)
    max_b = best['proxy_benefit']
    if max_b > 0:
        for fraction in sorted(set(p.cfg.coverage_targets + (p.cfg.chosen_target_fraction,))):
            if fraction <= 0 or fraction > 1: raise ValueError('Invalid coverage target.')
            sol = solve(p, 'min_cost_target', target=max_b*fraction)
            if sol is None:
                print(f'No feasible incumbent for {fraction:.0%} of best proxy benefit; skip (not proof of infeasibility).')
                continue
            sol['method'] = f'Minimum proxy collateral at {fraction:.0%} of feasible proxy maximum'
            points.append(sol)
    if p.cfg.chosen_policy == 'min_collateral_with_min_benefit':
        target = max_b * p.cfg.chosen_target_fraction
        matches = [z for z in points if z['mode']=='min_cost_target' and abs(z['target_proxy']-target)<max(1.,target*1e-8)]
        if matches:
            chosen = matches[0]
        else:
            chosen = dict(best)
            chosen['method'] = 'FALLBACK: max proxy benefit under safety budgets'
            chosen['status'] = 'Target-minimisation yielded no feasible incumbent; using safe max-proxy baseline. '+best['status']
    elif p.cfg.chosen_policy == 'penalty':
        chosen = next(z for z in points if z['mode']=='penalised' and z['penalty_weight']==p.cfg.chosen_lambda)
    else: raise ValueError('chosen_policy must be min_collateral_with_min_benefit or penalty')
    print(f'Optimisation scenarios: {len(points)}; elapsed {time.perf_counter()-start:.1f}s')
    return chosen, points

## 5. Independent all-currency retrospective evaluation and explanation writer

The following functions can use `is_laundering` **only after selection** to audit simulated transfer coverage. Never use audit outcomes to choose penalties if you intend to claim holdout performance. They do not prove transfers would have been prevented in real time.

In [ ]:
def selected_mask(d, selected):
    active = np.zeros(len(d.accounts), dtype=bool)
    if len(selected): active[np.asarray(selected,dtype=np.int32)] = True
    return active[d.sender] | active[d.receiver]


def retrospective_evaluate(d, selected):
    touched = selected_mask(d,selected)
    is_dirty = d.labelled.astype(bool)
    b = float(d.dollars[touched & is_dirty].sum())
    c = float(d.dollars[touched & ~is_dirty].sum())
    all_dirty = float(d.dollars[is_dirty].sum())
    bad_accounts = np.zeros(len(d.accounts), dtype=bool)
    bad_accounts[d.sender[is_dirty]] = True
    bad_accounts[d.receiver[is_dirty]] = True
    dirty_indices = np.flatnonzero(is_dirty)
    if len(dirty_indices):
        largest_dirty_idx = int(dirty_indices[np.argmax(d.dollars[dirty_indices])])
        remaining_total = all_dirty - d.dollars[largest_dirty_idx]
        remaining_blocked = b - (d.dollars[largest_dirty_idx] if touched[largest_dirty_idx] else 0.)
        pct_no_largest = 100*remaining_blocked/remaining_total if remaining_total > 0 else np.nan
    else: pct_no_largest = np.nan
    return {
        'observed_laundering_touched_usd':b,
        'observed_legitimate_touched_usd':c,
        'observed_legitimate_per_1m_laundering_usd':(1_000_000*c/b if b>0 else np.nan),
        'observed_laundering_value_coverage_pct':(100*b/all_dirty if all_dirty>0 else np.nan),
        'observed_laundering_transfer_count':int(np.sum(touched & is_dirty)),
        'observed_legitimate_transfer_count':int(np.sum(touched & ~is_dirty)),
        'observed_laundering_transfer_recall_pct':(100*np.sum(touched & is_dirty)/np.sum(is_dirty) if np.sum(is_dirty) else np.nan),
        'observed_laundering_value_pct_excluding_biggest_single_transfer':pct_no_largest,
        'observed_label_negative_frozen_accounts':int(np.sum(~bad_accounts[selected])),
        'largest_observed_legitimate_touched_usd':float(np.max(d.dollars[touched & ~is_dirty],initial=0.)),
        'largest_labelled_dirty_touched_usd':float(np.max(d.dollars[touched & is_dirty],initial=0.)),
    }


def meets_account_guards(p, selected):
    d, cfg = p.data, p.cfg
    nodes = np.asarray(selected, dtype=np.int32)
    if np.any(d.risks[nodes] < cfg.min_account_score): return False
    if cfg.max_account_gross_usd is not None and np.any(d.incident_gross[nodes] > cfg.max_account_gross_usd): return False
    if cfg.max_account_single_transfer_usd is not None and np.any(d.max_incident[nodes] > cfg.max_account_single_transfer_usd): return False
    if cfg.exclude_extreme_hubs and np.any(d.n_out[nodes] >= cfg.hub_outgoing_review): return False
    return True


def compare_baselines(d, p, prior_plan_path=None, prior_guardian_path=None):
    k = p.cfg.k
    n = len(d.accounts)
    total_degree = d.n_in+d.n_out
    b = np.minimum(d.dollars, p.benefit_cap)*p.risk_proxy
    base_b = incident_sum(d.sender,d.receiver,b,n)
    out = []
    for name,nodes in (
        ('Top account risk — all currency',np.argsort(-d.risks,kind='stable')[:k]),
        ('Highest degree — all currency',np.argsort(-total_degree,kind='stable')[:k]),
        ('Highest gross proxy flow — all currency',np.argsort(-base_b,kind='stable')[:k]),
    ):
        out.append(dict(method=name, selected=np.asarray(nodes,dtype=np.int32),
                        n_selected=len(nodes), origin_rows=len(nodes), unmatched_ids=0,
                        status='Unconstrained historical baseline — NOT required to obey v2 safeguards'))
    mapper = pd.Series(np.arange(n,dtype=np.int32),index=d.accounts)
    for fname, name in ((prior_plan_path,'Existing Step 3 plan — all currency'),
                        (prior_guardian_path,'Previous Guardian candidate — all currency')):
        if fname is None or not Path(fname).is_file(): continue
        prior = pd.read_csv(fname,usecols=['acct'])
        acct = prior.acct.dropna().astype(str).drop_duplicates().head(k)
        located = mapper.reindex(acct).dropna().to_numpy(dtype=np.int32)
        out.append(dict(method=name,selected=located,n_selected=len(located),
                        origin_rows=len(acct),unmatched_ids=len(acct)-len(located),
                        status='Retrospective matched-ID comparison; prior plan not rerun under v2 constraints'))
    return out


def make_recommendation_rows(p,selected):
    d = p.data
    if not len(selected):
        return pd.DataFrame(columns=['rank','acct','risk_score','est_money_blocked',
            'estimated_proxy_collateral','full_gross_incident_usd','marginal_gross_touched_usd',
            'outgoing_tx','incoming_tx','reason_facts','requires_human_approval'])
    # Explanation rank sorted by *model's* individual suspicious proxy, not by labels.
    raw_benefit = np.minimum(d.dollars,p.benefit_cap)*p.risk_proxy
    c = d.dollars * np.maximum(1-p.risk_proxy,p.cfg.min_proxy_legitimate_fraction)
    scores = incident_sum(d.sender,d.receiver,raw_benefit,len(d.accounts))
    ordering = sorted(map(int,selected),key=lambda idx:(-scores[idx],d.accounts[idx]))
    seen = np.zeros(len(d.dollars),bool)
    rows=[]
    for rank,node in enumerate(ordering,1):
        inc = (d.sender==node)|(d.receiver==node)
        added = inc & ~seen
        seen |= inc
        neigh = np.unique(np.r_[d.receiver[d.sender==node],d.sender[d.receiver==node]])
        cross = int(np.count_nonzero(inc&(d.from_bank!=d.to_bank)))
        marginal_b=float(raw_benefit[added].sum()); marginal_c=float(c[added].sum())
        marginal_g=float(d.dollars[added].sum())
        summary = (f'Risk score {d.risks[node]:.3f} (account-level; not transaction probability); '
          f'{int(inc.sum()):,} historical transfers across all currencies; {len(neigh):,} counterparties; '
          f'{cross:,} cross-bank transfers; {marginal_b:,.0f} capped USD suspicion proxy, '
          f'{marginal_c:,.0f} USD uncalibrated collateral proxy, '
          f'{marginal_g:,.0f} USD full incremental historical exposure; HUMAN REVIEW ONLY.')
        rows.append(dict(rank=rank,acct=d.accounts[node],risk_score=float(d.risks[node]),
                         est_money_blocked=marginal_b,estimated_proxy_collateral=marginal_c,
                         full_gross_incident_usd=float(d.incident_gross[node]),
                         marginal_gross_touched_usd=marginal_g,
                         outgoing_tx=int(d.n_out[node]),incoming_tx=int(d.n_in[node]),
                         reason_facts=summary,requires_human_approval=True))
    return pd.DataFrame(rows)


def by_currency_audit(d, selected):
    touched = selected_mask(d,selected)
    subset = pd.DataFrame({'currency':d.currency[touched],
                           'is_laundering':d.labelled[touched],
                           'amount_original':d.original_amount[touched],
                           'amount_converted_usd':d.dollars[touched]})
    if subset.empty:
        return pd.DataFrame(columns=['currency','is_laundering','transactions','amount_original','amount_usd'])
    return subset.groupby(['currency','is_laundering'],as_index=False).agg(
        transactions=('amount_original','size'),
        amount_original=('amount_original','sum'),
        amount_usd=('amount_converted_usd','sum'))


def audit_marginals_only(d, plan):
    seen=np.zeros(len(d.dollars),bool)
    out=[]
    lookup={acct:i for i,acct in enumerate(d.accounts)}
    for row in plan.itertuples(index=False):
        idx=lookup[row.acct]
        touched=(d.sender==idx)|(d.receiver==idx)
        marginal=touched & ~seen
        seen |= touched
        is_dirty = d.labelled==1
        out.append({'rank':row.rank,'acct':row.acct,
                    'observed_labelled_laundering_marginal_usd':float(d.dollars[marginal&is_dirty].sum()),
                    'observed_labelled_legitimate_marginal_usd':float(d.dollars[marginal&~is_dirty].sum())})
    return pd.DataFrame(out,columns=['rank','acct','observed_labelled_laundering_marginal_usd',
                                     'observed_labelled_legitimate_marginal_usd'])

## 6. Correctness tests — run before real-data optimisation

These are deliberately tiny tests: self-transfer and overlapping endpoint union, collateral with **full** USD instead of capped amount, missing conversion errors, and **selection unchanged when labels change**. They are not substitutes for validating the real upstream files.

In [ ]:
def self_tests():
    # One transfer from account 0 to itself must add exactly once.
    np.testing.assert_allclose(incident_sum(np.array([0]),np.array([0]),np.array([17.]),2),[17,0])
    # Accounts 0 and 1 selected touch only two distinct transfers, not three.
    f=np.array([0,0,2]);t=np.array([1,0,1]);a=np.array([10.,20.,30.])
    d=Data(np.array(['A','B','C']),np.array([.9,.9,.9]),f,t,a,a,
           np.array(['US Dollar']*3),np.array([0,1,1],np.int8),np.array(['2026-01-01']*3),
           np.array(['B1']*3),np.array(['B2']*3),np.arange(3),
           np.bincount(f,minlength=3),np.bincount(t,minlength=3),
           incident_sum(f,t,a,3),np.array([20.,30.,30.]),3,3,dict(FX_TO_USD))
    np.testing.assert_allclose(selected_mask(d,np.array([0,1])),[True,True,True])
    assert d.dollars[selected_mask(d,np.array([0,1]))].sum()==60.
    cfg=replace(Config(),max_account_gross_usd=None,max_account_single_transfer_usd=None,
                max_portfolio_gross_usd=100.,max_portfolio_proxy_collateral_usd=None,
                max_portfolio_touched_tx=None,min_account_score=0.)
    p=build_problem(d,cfg)
    before = (p.candidates.copy(),p.group_benefit.copy(),p.group_cost.copy())
    d2=replace(d,labelled=np.array([1,0,0],np.int8))
    p2=build_problem(d2,cfg)
    np.testing.assert_array_equal(before[0],p2.candidates)
    np.testing.assert_allclose(before[1],p2.group_benefit)
    np.testing.assert_allclose(before[2],p2.group_cost)
    # All selected retrospective totals count each transaction once.
    aa=retrospective_evaluate(d,np.array([0,1]));assert aa['observed_laundering_touched_usd']==50.
    # Max gross cap can't be evaded by applying the clipping rule.
    assert not meets_limits(p,(0.,0.,110.,0))
    # Original array includes two currencies; no addition without conversion.
    assert FX_TO_USD['Brazil Real']>0 and FX_TO_USD['Brazil Real']!=1.
    print('Self-tests passed: endpoint union, self-transfer, label independence, gross guard, FX definitions.')

self_tests()

## 7. Run the experiment and save **separate** audit and candidate files

The chosen development policy is fixed by `CFG.chosen_policy` and its stated minimum **proxy** coverage target. Compare methods on identical **full-currency** transactions. Reports are retrospective and label-derived only where their filenames contain `EVAL_ONLY`. Never present the exploratory winner as unseen-test performance or claim actual funds stopped.

In [ ]:
def file_sha256(path, chunk_size=1024*1024):
    h=hashlib.sha256()
    with Path(path).open('rb') as fp:
        for buf in iter(lambda:fp.read(chunk_size),b''):
            h.update(buf)
    return h.hexdigest()


def run_experiment(tx_path, risk_path, outdir, cfg, old_plan=None, old_guardian=None):
    start=time.perf_counter()
    d=load_data(tx_path,risk_path)
    p=build_problem(d,cfg)
    chosen, scenarios=run_frontier(p)
    baselines=compare_baselines(d,p,old_plan,old_guardian)
    out=Path(outdir);out.mkdir(parents=True,exist_ok=True)
    rows=[]
    for sol in scenarios+baselines:
        observed=retrospective_evaluate(d,sol['selected'])
        if 'proxy_benefit' in sol:
            predicted=(sol['proxy_benefit'],sol['proxy_collateral'],sol['gross_touched'],sol['tx_touched'])
        else:
            # Baselines may violate new safety caps; report, do not disguise.
            selected=sol['selected']
            full=selected_mask(d,selected)
            prob=p.risk_proxy
            pred_b=np.minimum(d.dollars,p.benefit_cap)*prob
            pred_c=d.dollars*np.maximum(1-prob,cfg.min_proxy_legitimate_fraction)
            predicted=(float(pred_b[full].sum()),float(pred_c[full].sum()),
                       float(d.dollars[full].sum()),int(full.sum()))
        rows.append(dict(method=sol['method'],n_selected=len(sol['selected']),
                         selected_account_ids=';'.join(d.accounts[sol['selected']].tolist()),
                         proxy_benefit_usd=predicted[0],proxy_collateral_usd=predicted[1],
                         full_gross_touched_usd=predicted[2],tx_touched=predicted[3],
                         within_v2_limits=(meets_limits(p,predicted) and meets_account_guards(p,sol['selected']) and len(sol['selected'])<=cfg.k),
                         solver_status=sol.get('status',''),mip_gap=sol.get('mip_gap'),
                         runtime_seconds=sol.get('runtime_seconds'),
                         missing_prior_accounts=sol.get('unmatched_ids',0),
                         **observed))
    comparison=pd.DataFrame(rows)
    candidates=make_recommendation_rows(p,chosen['selected'])
    currencies=by_currency_audit(d,chosen['selected'])
    marginal=audit_marginals_only(d,candidates)
    comparison.to_csv(out/'all_currency_comparison_EVAL_ONLY.csv',index=False)
    candidates.to_csv(out/'freeze_plan_candidate.csv',index=False)
    currencies.to_csv(out/'selected_currency_audit_EVAL_ONLY.csv',index=False)
    marginal.to_csv(out/'candidate_label_marginals_EVAL_ONLY.csv',index=False)
    plan_recomputed=selected_mask(d,chosen['selected'])
    assert np.isclose(d.dollars[plan_recomputed].sum(), chosen['gross_touched'], rtol=1e-8,atol=0.1)
    assert len(candidates)==chosen['n_selected']
    assert len(candidates['acct'].unique())==chosen['n_selected']
    assert np.isclose(currencies.amount_usd.sum(),chosen['gross_touched'],rtol=1e-8,atol=0.1)
    out_manifest={
        'mode':'RETROSPECTIVE, SYNTHETIC DEVELOPMENT AUDIT; NOT PROSPECTIVE OR CAUSAL',
        'selection_uses_ground_truth_labels':False,
        'scores_are_calibrated_transaction_probabilities':False,
        'fx_pairing_requires_upstream_verification':'Confirm amount_paid means Amount Paid AND currency is Payment Currency',
        'usd_fx_rates_approximate':d.fx_rates,
        'dataset_rows':d.tx_file_rows,'risk_rows':d.risk_file_rows,
        'all_currencies_included':sorted(set(d.currency.tolist())),
        'input_hash_sha256':{'transactions':file_sha256(tx_path),'risk':file_sha256(risk_path)},
        'config':asdict(cfg),'shortlisted_candidates':len(p.candidates),
        'candidate_restrictions_eligible_accounts':p.eligible_count,
        'coverage_groups':len(p.group_nodes),'chosen_method':chosen['method'],
        'chosen_solver_status':chosen['status'],'chosen_mip_gap':chosen['mip_gap'],
        'chosen_selected_count':len(chosen['selected']),
        'chosen_within_safety_limits':meets_limits(p,plan_numbers(p,chosen['selected'])),
        'scipy_milp_available':HAVE_MILP,'run_seconds':time.perf_counter()-start,
        'output_directory':str(out.resolve()),
        'no_freezes_executed':True,'manual_authorisation_required':True,
        'limitations':[
            'account risk scores are full-snapshot and may leak future information',
            'synthetic input slice may have been selected using ground-truth labels',
            'this is account-freeze coverage, not a tested targeted-payment-hold workflow',
            'amounts are historical transaction sums, not recovered proceeds or customer balances',
            'USD conversion uses time-invariant approximate FX rates',
            'no performance claims on an untouched chronological test period',
            'solver optimality is only within the shortlisted candidate set and proxy model',
            'collateral proxy is not a calibrated estimate of actual benign transaction value',
        ],
    }
    (out/'run_manifest.json').write_text(json.dumps(out_manifest,indent=2,default=str),encoding='utf-8')
    return dict(data=d,problem=p,chosen=chosen,scenarios=scenarios,comparison=comparison,
                freeze_plan=candidates,currency_audit=currencies,manifest=out_manifest,outdir=out)

In [ ]:
# The single intended execution cell.
result = run_experiment(INPUT_PATH,RISK_PATH,OUTPUT_DIR,CFG,
                        old_plan=PRIOR_PLAN_PATH if PRIOR_PLAN_PATH.is_file() else None,
                        old_guardian=PRIOR_GUARDIAN_PATH if PRIOR_GUARDIAN_PATH.is_file() else None)
print('\nRETROSPECTIVE ALL-CURRENCY EVALUATION (NOT live savings):')
cols = ['method','n_selected','observed_laundering_touched_usd',
        'observed_legitimate_touched_usd','observed_legitimate_per_1m_laundering_usd',
        'observed_laundering_transfer_count','observed_legitimate_transfer_count',
        'within_v2_limits']
print(result['comparison'][cols].to_string(index=False,float_format=lambda v:f'{v:,.2f}'))
print('\nSelected development policy:',result['manifest']['chosen_method'])
print('MILP status:',result['chosen']['status'])
print('Accounts recommended:',result['chosen']['n_selected'])
print('\nCURRENCY AUDIT — original units cannot be summed across currencies:')
print(result['currency_audit'].to_string(index=False,float_format=lambda v:f'{v:,.2f}'))
print('\nRecommendation rows (no true labels):')
display(result['freeze_plan'][['rank','acct','risk_score','est_money_blocked',
    'estimated_proxy_collateral','marginal_gross_touched_usd','reason_facts']])
print('\nOutput files saved to:',OUTPUT_DIR.resolve())
print('Original /shared/freeze_plan.csv was NOT changed.')

## 8. Plot the honest all-currency trade-off

The first chart plots **model proxies**, not predicted fraud probabilities. The second audits historical labels. Both charts reflect the same fully converted dataset. Do **not** tune the final policy using the second chart and then call that an unbiased test.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('Matplotlib is not available; CSV outputs still work.')
else:
    temp=result['comparison']
    dev=temp[temp.method.str.startswith('Proxy penalty ')].copy()
    fig,ax=plt.subplots(figsize=(8,5))
    ax.scatter(dev.proxy_collateral_usd/1e6,dev.proxy_benefit_usd/1e6)
    for r in dev.itertuples():
        ax.annotate(r.method.replace('Proxy penalty ','λ=').replace(' (constrained)',''),
                    (r.proxy_collateral_usd/1e6,r.proxy_benefit_usd/1e6),fontsize=8)
    ax.set(xlabel='Uncalibrated proxy collateral (USD millions)',
           ylabel='Capped proxy suspicious benefit (USD millions)',
           title='ALL-CURRENCY development trade-off (proxy only)')
    ax.grid(alpha=.25);fig.tight_layout()
    fig.savefig(OUTPUT_DIR/'all_currency_proxy_frontier.png',dpi=135)
    plt.show();plt.close(fig)
    fig,ax=plt.subplots(figsize=(8,5))
    ax.scatter(dev.observed_legitimate_touched_usd/1e6,
               dev.observed_laundering_touched_usd/1e6)
    for r in dev.itertuples():
        ax.annotate(r.method.replace('Proxy penalty ','λ=').replace(' (constrained)',''),
                    (r.observed_legitimate_touched_usd/1e6,r.observed_laundering_touched_usd/1e6),fontsize=8)
    ax.set(xlabel='Historical labelled legitimate value touched (USD millions)',
           ylabel='Historical labelled laundering value touched (USD millions)',
           title='ALL-CURRENCY historical label audit — NOT a future test')
    ax.grid(alpha=.25);fig.tight_layout()
    fig.savefig(OUTPUT_DIR/'all_currency_label_audit_EVAL_ONLY.png',dpi=135)
    plt.show();plt.close(fig)

In [ ]:

import numpy as np
import pandas as pd

d = result["data"]
acct = "A_1abc58e5f795"

matches = np.flatnonzero(d.accounts == acct)

if len(matches) == 0:
    print("Account not found in current dataset.")
else:
    idx = int(matches[0])
    incoming = d.receiver == idx
    outgoing = d.sender == idx
    touched = incoming | outgoing

    for name, mask in [
        ("Incoming", incoming),
        ("Outgoing", outgoing),
        ("All activity", touched)
    ]:
        dates = pd.to_datetime(d.timestamp[mask], utc=True)

        print(f"\n{name}:")
        print("Transactions:", int(mask.sum()))
        print("Total USD:", f"${d.dollars[mask].sum():,.2f}")
        print("First:", dates.min())
        print("Last:", dates.max())

    print("\nSelf-transfers:",
          int((incoming & outgoing).sum()))

    print("\nOutgoing value by currency:")
    report = pd.DataFrame({
        "currency": d.currency[outgoing],
        "usd": d.dollars[outgoing]
    })
    display(
        report.groupby("currency")
        .agg(transfers=("usd", "size"),
             total_usd=("usd", "sum"))
        .sort_values("total_usd", ascending=False)
    )

    print("\nSelected in current V2 plan:",
          acct in set(result["freeze_plan"]["acct"]))


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib

S = Path("/home/ibm-user/shared")
V = S / "guardian_v2_outputs"

files = {
    "Transactions": S / "clean_transactions.csv",
    "Risk scores": S / "account_risk.csv",
    "V2 recommendations": V / "freeze_plan_candidate.csv",
    "V2 manifest": V / "run_manifest.json",
    "V2 evaluation": V / "all_currency_comparison_EVAL_ONLY.csv",
    "Ahmed explanations": S / "freeze_plan_explained.csv",
}

failures = []

def check(name, condition):
    passed = bool(condition)
    print(("PASS" if passed else "FAIL") + " | " + name)
    if not passed:
        failures.append(name)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("=== 1. FILE CHECK ===")
for name, path in files.items():
    check(name + " exists", path.is_file())
    print("  ", path)

print("\n=== 2. INPUT AND V2 CHECKS ===")

required = ["Transactions", "Risk scores",
            "V2 recommendations", "V2 manifest",
            "V2 evaluation"]

if all(files[x].is_file() for x in required):

    tx = pd.read_csv(files["Transactions"], usecols=[
        "tx_id", "from_acct", "to_acct",
        "from_bank", "to_bank",
        "currency", "is_laundering"
    ])
    risk = pd.read_csv(files["Risk scores"])
    plan = pd.read_csv(files["V2 recommendations"])
    comparison = pd.read_csv(files["V2 evaluation"])

    with open(files["V2 manifest"]) as f:
        manifest = json.load(f)

    accounts = set(tx["from_acct"]) | set(tx["to_acct"])
    risk_accounts = set(risk["acct"])

    print("Transactions:", len(tx))
    print("Unique accounts:", len(accounts))
    print("Currencies:", tx["currency"].nunique())
    print("Laundering transactions:",
          int((tx["is_laundering"] == 1).sum()))

    check("Transaction IDs unique",
          tx["tx_id"].is_unique)
    check("100% account risk coverage",
          accounts.issubset(risk_accounts))
    check("Risk account IDs unique",
          risk["acct"].is_unique)
    check("Risk scores valid",
          pd.to_numeric(
              risk["risk_score"], errors="coerce"
          ).between(0, 1).all())

    check("All 15 currencies represented",
          tx["currency"].nunique() == 15)

    banks = set(tx["from_bank"]) | set(tx["to_bank"])
    check("Three simulated banks represented",
          len(banks) == 3)

    print("\n=== 3. VERSION MATCH ===")
    saved_hashes = manifest.get("input_hash_sha256", {})

    check("V2 used current transactions",
          sha256(files["Transactions"]) ==
          saved_hashes.get("transactions"))

    check("V2 used current risk scores",
          sha256(files["Risk scores"]) ==
          saved_hashes.get("risk"))

    check("V2 safety limits passed",
          manifest.get("chosen_within_safety_limits") is True)

    check("V2 includes all currencies",
          set(manifest.get("all_currencies_included", []))
          == set(tx["currency"]))

    print("\n=== 4. RECOMMENDATION CHECK ===")

    expected = {
        "rank", "acct", "risk_score",
        "est_money_blocked",
        "estimated_proxy_collateral",
        "reason_facts"
    }

    check("V2 columns correct",
          expected.issubset(plan.columns))
    check("No duplicate recommendations",
          plan["acct"].is_unique)
    check("All recommended accounts exist",
          set(plan["acct"]).issubset(accounts))
    check("No more than 10 recommendations",
          0 < len(plan) <= 10)
    check("Ranks sequential",
          plan["rank"].tolist() ==
          list(range(1, len(plan) + 1)))

    chosen_method = manifest.get("chosen_method")
    matching = comparison[
        comparison["method"] == chosen_method
    ]

    check("Chosen policy in evaluation report",
          len(matching) == 1)

    if len(matching) == 1:
        row = matching.iloc[0]
        ids = set(str(
            row["selected_account_ids"]
        ).split(";"))

        check("Evaluation matches V2 account IDs",
              ids == set(plan["acct"]))

        check("Chosen evaluation within V2 limits",
              str(row["within_v2_limits"]).lower()
              == "true")

    print("Selected accounts:", len(plan))
    print("Selected policy:", chosen_method)

    print("\n=== 5. AHMED INTEGRATION ===")

    if files["Ahmed explanations"].is_file():
        explained = pd.read_csv(
            files["Ahmed explanations"]
        )

        check("Ahmed output has acct column",
              "acct" in explained.columns)

        if "acct" in explained.columns:
            check("Ahmed explains all V2 accounts",
                  set(explained["acct"]) ==
                  set(plan["acct"]))

            check("Ahmed has one row per account",
                  len(explained) == len(plan)
                  and explained["acct"].is_unique)

            check("Ahmed output is not older than V2",
                  files["Ahmed explanations"].stat().st_mtime
                  >= files["V2 recommendations"].stat().st_mtime)

    print("\n=== 6. DATA WARNINGS ===")
    print("Self-transfers:",
          int((tx["from_acct"] == tx["to_acct"]).sum()))

    legacy = S / "freeze_plan.csv"

    if legacy.is_file():
        old = pd.read_csv(legacy)
        print("Old freeze_plan.csv equals V2 account set:",
              set(old["acct"]) == set(plan["acct"]))
        print("Note: Ahmed must use V2, not an old plan.")

print("\n" + "=" * 55)

if failures:
    print("CHECKS NEED ATTENTION:", len(failures))
    for item in failures:
        print(" -", item)
else:
    print("ALL AUTOMATED FILE AND HANDOFF CHECKS PASSED")

print("=" * 55)


## 9. Handoff and acceptance criteria

**Do not replace the team `freeze_plan.csv` yet.** Review `guardian_v2_outputs/freeze_plan_candidate.csv` and its all-currency audit with your friend and Ahmed. `candidate_label_marginals_EVAL_ONLY.csv` and `all_currency_comparison_EVAL_ONLY.csv` contain true labels for evaluation only; do not pass those fields to a decision-making model or Granite as if known when freezing.

**A good result requires:** acceptable labelled legitimate value and count affected; useful laundering value **and transaction-count** coverage; gross/proxy-cost budgets genuinely obeyed; same all-currency evaluation for baselines; good robustness without a handful of whales; and eventually untouched future-period validation. A plan with no freezes has zero disruption **and zero laundering benefit**. No one can guarantee zero legitimate harm from uncalibrated scores.

Known remaining limitations: (1) model predictions and dataset sampling may leak labels/future; (2) exchange rates are approximate and unverified against raw Payment Currency; (3) hard portfolio limits are retrospective, not presently enforceable in a real-time decision process; (4) targeted payment holds and a causal real-time replay are not yet implemented; (5) MILP shortlist optimality does not imply a globally optimal or safe real-world freeze plan.